# Entity analysis — all classes × models

Loops over every `addmypet_by_model/<class>/<model>/` subfolder, counts how many species
have each DEB entity group, and saves one `summary_<CLASS>_<MODEL>.json` per subfolder.

In [ ]:
import json
import os
import glob
import pandas as pd

BASE_DIR = "../addmypet_by_model"
OUT_DIR  = "../summary_classes"

os.makedirs(OUT_DIR, exist_ok=True)

# ── entity groups ──────────────────────────────────────────────────────────────
GROUPS = [
    ("ab",      ["ab"]),
    ("ah",      ["ah"]),
    ("am",      ["am"]),
    ("aj / tj", ["aj", "tj"]),
    ("tg",      ["tg"]),
    ("ax / tx", ["ax", "tx"]),
    ("as / ts", ["as", "ts"]),
    ("ap / tp", ["ap", "tp"]),
    ("aR / tR", ["aR", "tR"]),
    ("ae / te", ["ae", "te"]),
]

def match(e, keys):
    return any(
        e == k or e.startswith(k + "_") or (e.startswith(k) and len(e) > len(k))
        for k in keys
    )

# ── discover all class/model subfolders ────────────────────────────────────────
subfolders = []
for cls in sorted(os.listdir(BASE_DIR)):
    cls_path = os.path.join(BASE_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    for model in sorted(os.listdir(cls_path)):
        model_path = os.path.join(cls_path, model)
        if os.path.isdir(model_path):
            subfolders.append((cls, model, model_path))

print(f"Subfolders found: {len(subfolders)}")
for cls, model, _ in subfolders:
    print(f"  {cls}/{model}")

In [2]:
# ── analyse one class/model subfolder and save summary ────────────────────────
def analyse_subfolder(cls, model, folder):
    species_entities = []
    for fp in sorted(glob.glob(os.path.join(folder, "*.json"))):
        with open(fp) as f:
            data = json.load(f)
        species_entities.append([r["entity"] for r in data if "entity" in r])

    total = len(species_entities)

    entity_summary = []
    for label, keys in GROUPS:
        n_any = sum(1 for ents in species_entities if any(match(e, keys) for e in ents))
        entity_summary.append({
            "entity":   label,
            "with_any": n_any,
            "with_no":  total - n_any,
        })

    return {
        "class":          cls,
        "deb_model":      model,
        "total_species":  total,
        "entity_summary": entity_summary,
    }

# ── run over all subfolders and save one summary per class/model ───────────────
all_summaries = []

for cls, model, folder in subfolders:
    result = analyse_subfolder(cls, model, folder)

    # save as summary_<CLASS>_<MODEL>.json inside OUT_DIR/<CLASS>/
    cls_out = os.path.join(OUT_DIR, cls)
    os.makedirs(cls_out, exist_ok=True)
    out_file = os.path.join(cls_out, f"summary_{cls}_{model}.json")
    with open(out_file, "w") as f:
        json.dump(result, f, indent=2)

    all_summaries.append(result)
    print(f"  OK  {cls}/{model:<6}  species={result['total_species']:5d}  -> {out_file}")

print(f"\nDone. {len(all_summaries)} summaries written to {OUT_DIR}")

  OK  Acoelomorpha/abj     species=    1  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Acoelomorpha/summary_Acoelomorpha_abj.json
  OK  Actinopterygii/abj     species= 2386  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Actinopterygii/summary_Actinopterygii_abj.json
  OK  Actinopterygii/std     species=  388  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Actinopterygii/summary_Actinopterygii_std.json
  OK  Amphibia/std     species=  246  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Amphibia/summary_Amphibia_std.json
  OK  Anopla/abj     species=    1  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Anopla/summary_Anopla_abj.json
  OK  Anthozoa/std     species=    3  -> /Users/somaya/Documents/Phd/dataset_generation_NER/summary_classes/Anthozoa/summary_Anthozoa_std.json
  OK  Appendicularia/std     species=    3  -> /Users/somaya/Documents/Phd/dataset_generation_NER/su

In [3]:
# ── entity counts table: one row per class/model ──────────────────────────────
rows = []
for s in all_summaries:
    row = {"class": s["class"], "model": s["deb_model"], "total_species": s["total_species"]}
    for ent in s["entity_summary"]:
        row[ent["entity"]] = ent["with_any"]
    rows.append(row)

df = pd.DataFrame(rows).set_index(["class", "model"])
print(df.to_string())

                           total_species    ab  ah    am  aj / tj    tg  ax / tx  as / ts  ap / tp  aR / tR  ae / te
class               model                                                                                           
Acoelomorpha        abj                1     1   0     1        0     0        0        0        1        0        0
Actinopterygii      abj             2386  1168  47  2386       37     0        0        0      288        1        0
                    std              388   176  20   372        1     0        0        0       77        0        0
Amphibia            std              246   191   6   245      122     0        0        0      196        1        0
Anopla              abj                1     0   0     1        1     0        0        0        1        0        0
Anthozoa            std                3     0   0     3        0     0        0        0        1        0        0
Appendicularia      std                3     2   0     3        

In [4]:
# ── detailed entity breakdown per class/model ─────────────────────────────────
def detailed_breakdown(cls, model, folder):
    results = []
    for fp in sorted(glob.glob(os.path.join(folder, "*.json"))):
        with open(fp) as f:
            data = json.load(f)
        species  = os.path.basename(fp).replace(".json", "")
        entities = [r["entity"] for r in data if "entity" in r]
        results.append({"species": species, "entities": entities})

    print(f"\n{'='*70}")
    print(f"CLASS: {cls}  MODEL: {model}  ({len(results)} species)")
    print(f"{'='*70}")

    for label, keys in GROUPS:
        n_exact    = sum(1 for r in results if any(e in keys for e in r["entities"]))
        n_suf_only = sum(1 for r in results
                         if not any(e in keys for e in r["entities"])
                         and any(match(e, keys) for e in r["entities"]))
        n_both     = sum(1 for r in results
                         if any(e in keys for e in r["entities"])
                         and any(e not in keys and match(e, keys) for e in r["entities"]))
        n_any  = sum(1 for r in results if any(match(e, keys) for e in r["entities"]))
        n_none = len(results) - n_any
        print(f"  {label:10s}  exact={n_exact:4d}  suf_only={n_suf_only:4d}  both={n_both:4d}  any={n_any:4d}  none={n_none:4d}")

for cls, model, folder in subfolders:
    detailed_breakdown(cls, model, folder)


CLASS: Acoelomorpha  MODEL: abj  (1 species)
  ab          exact=   1  suf_only=   0  both=   0  any=   1  none=   0
  ah          exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  am          exact=   1  suf_only=   0  both=   0  any=   1  none=   0
  aj / tj     exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  tg          exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  ax / tx     exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  as / ts     exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  ap / tp     exact=   1  suf_only=   0  both=   0  any=   1  none=   0
  aR / tR     exact=   0  suf_only=   0  both=   0  any=   0  none=   1
  ae / te     exact=   0  suf_only=   0  both=   0  any=   0  none=   1

CLASS: Actinopterygii  MODEL: abj  (2386 species)
  ab          exact=1158  suf_only=  10  both=   3  any=1168  none=1218
  ah          exact=  41  suf_only=   6  both=   1  any=  47  none=2339
  am          exact=2385  suf_only=   1

## Summary: species and models per class

Shows the total species per class broken down by model, loaded from the summaries above.

In [5]:
# ── species per class broken down by model ────────────────────────────────────
from collections import defaultdict

class_model_counts = defaultdict(dict)
for s in all_summaries:
    class_model_counts[s["class"]][s["deb_model"]] = s["total_species"]

print(f"{'Class':<30} {'Model':<8} {'Species':>8}")
print("-" * 50)
for cls in sorted(class_model_counts):
    counts = class_model_counts[cls]
    total  = sum(counts.values())
    for i, (model, n) in enumerate(sorted(counts.items())):
        label = cls if i == 0 else ""
        print(f"{label:<30} {model:<8} {n:>8}")
    if len(counts) > 1:
        print(f"{'':30} {'TOTAL':<8} {total:>8}")
    print()

Class                          Model     Species
--------------------------------------------------
Acoelomorpha                   abj             1

Actinopterygii                 abj          2386
                               std           388
                               TOTAL        2774

Amphibia                       std           246

Anopla                         abj             1

Anthozoa                       std             3

Appendicularia                 std             3

Arachnida                      abj             7

Archiacanthocephala            std             1

Ascidiacea                     std             9

Asteroidea                     abj            11

Aves                           std          1744

Bivalvia                       abj           134

Branchiopoda                   abj            30
                               std            77
                               TOTAL         107

Cephalaspidomorphi             std            14

Ceph

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

output_md  = "../../entity_analysis.md"
output_png = "../../entity_analysis.png"
output_svg = "../../entity_analysis.svg"

entity_labels = [g[0] for g in GROUPS]

# group by class
by_class = defaultdict(list)
for s in all_summaries:
    by_class[s["class"]].append(s)

# ── markdown (HTML table with sub-columns) ────────────────────────────────────
lines = ["# Entity Analysis per Class per Model\n\n"]
lines.append("<table>\n<thead>\n")
lines.append("  <tr>\n")
lines.append("    <th rowspan='2'>Class</th>\n")
lines.append("    <th rowspan='2'>Model</th>\n")
lines.append("    <th rowspan='2'>Total</th>\n")
for e in entity_labels:
    lines.append(f"    <th colspan='2'>{e}</th>\n")
lines.append("  </tr>\n  <tr>\n")
for _ in entity_labels:
    lines.append("    <th>with</th><th>without</th>\n")
lines.append("  </tr>\n</thead>\n<tbody>\n")

for cls in sorted(by_class):
    models = sorted(by_class[cls], key=lambda x: x["deb_model"])
    for s in models:
        model  = s["deb_model"]
        total  = s["total_species"]
        counts = {e["entity"]: e for e in s["entity_summary"]}
        lines.append("  <tr>\n")
        lines.append(f"    <td>{cls}</td><td>{model}</td><td>{total}</td>\n")
        for e in entity_labels:
            lines.append(f"    <td>{counts[e]['with_any']}</td><td>{counts[e]['with_no']}</td>\n")
        lines.append("  </tr>\n")
    if len(models) > 1:
        cls_total = sum(s["total_species"] for s in models)
        lines.append("  <tr style='font-weight:bold;background:#f0f0f0'>\n")
        lines.append(f"    <td>{cls} total</td><td>—</td><td>{cls_total}</td>\n")
        for e in entity_labels:
            idx = entity_labels.index(e)
            lines.append(f"    <td>{sum(s['entity_summary'][idx]['with_any'] for s in models)}</td>"
                         f"<td>{sum(s['entity_summary'][idx]['with_no'] for s in models)}</td>\n")
        lines.append("  </tr>\n")

lines.append("</tbody>\n</table>\n")

with open(output_md, "w") as f:
    f.writelines(lines)
print(f"Saved {output_md}")

# ── heatmap: % coverage per entity per class/model ────────────────────────────
row_labels = [f"{s['class']} / {s['deb_model']}" for s in all_summaries]
data = np.array([
    [(e["with_any"] / s["total_species"] * 100) if s["total_species"] > 0 else 0
     for e in s["entity_summary"]]
    for s in all_summaries
])

fig, ax = plt.subplots(figsize=(len(entity_labels) * 1.2 + 2, len(row_labels) * 0.38 + 2))

cmap = plt.cm.YlOrRd.copy()
im = ax.imshow(data, cmap=cmap, aspect="auto", vmin=0, vmax=100)

for i in range(len(row_labels)):
    for j in range(len(entity_labels)):
        v = data[i, j]
        if v > 0:
            txt_color = "white" if v > 60 else "black"
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center", fontsize=7, color=txt_color)

ax.set_xticks(range(len(entity_labels)))
ax.set_xticklabels(entity_labels, fontsize=9, fontweight="bold")
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=7)
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")

cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label("% species with entity", fontsize=8)

ax.set_title("Entity Coverage per Class / Model  (% of species)", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()

fig.savefig(output_png, dpi=150, bbox_inches="tight")
fig.savefig(output_svg, bbox_inches="tight")
plt.close()
print(f"Saved {output_png}")
print(f"Saved {output_svg}")


In [ ]:

# ── collect all summary files into a single flat folder (no class subfolders) ──
import os
import glob
import shutil

OUT_DIR      = "../summary_classes"
FLAT_OUT_DIR = "../summary_no_class"
os.makedirs(FLAT_OUT_DIR, exist_ok=True)

copied = []
for fp in sorted(glob.glob(os.path.join(OUT_DIR, "**", "summary_*.json"), recursive=True)):
    dest = os.path.join(FLAT_OUT_DIR, os.path.basename(fp))
    shutil.copy2(fp, dest)
    copied.append(os.path.basename(fp))

print(f"Copied {len(copied)} summary files to {FLAT_OUT_DIR}")
for name in copied:
    print(f"  {name}")

In [ ]:

# ── organise flat summaries into per-model folders, max 11 files each ──────────
import os
import glob
import shutil
import math
from collections import defaultdict

FLAT_OUT_DIR    = "../summary_no_class"
MODELS_OUT_DIR  = "../life_cycle/summary_by_model"
MAX_PER_FOLDER  = 12
SPLIT_MODELS    = {"abj", "std"}  # only these get batched subfolders

# group files by DEB model (last underscore-delimited token before .json)
by_model = defaultdict(list)
for fp in sorted(glob.glob(os.path.join(FLAT_OUT_DIR, "summary_*.json"))):
    model = os.path.basename(fp).replace(".json", "").rsplit("_", 1)[-1]
    by_model[model].append(fp)

# collect all non-split models into a single "other" folder
other_files = []
for model, files in sorted(by_model.items()):
    if model not in SPLIT_MODELS:
        other_files.extend(files)

if other_files:
    other_path = os.path.join(MODELS_OUT_DIR, "other")
    os.makedirs(other_path, exist_ok=True)
    for fp in other_files:
        shutil.copy2(fp, os.path.join(other_path, os.path.basename(fp)))
    print(f"  {'other':<12}  {len(other_files):2d} files")

# batch abj and std into numbered subfolders
for model in sorted(SPLIT_MODELS):
    files = by_model.get(model, [])
    if not files:
        continue
    n_batches = math.ceil(len(files) / MAX_PER_FOLDER)
    for batch_idx, start in enumerate(range(0, len(files), MAX_PER_FOLDER), 1):
        batch = files[start : start + MAX_PER_FOLDER]
        folder_name = model if n_batches == 1 else f"{model}_{batch_idx}"
        folder_path = os.path.join(MODELS_OUT_DIR, folder_name)
        os.makedirs(folder_path, exist_ok=True)
        for fp in batch:
            shutil.copy2(fp, os.path.join(folder_path, os.path.basename(fp)))
        print(f"  {folder_name:<12}  {len(batch):2d} files")

print(f"\nDone. Organised into {MODELS_OUT_DIR}")